# KnowledgeHub RAG v0.5 – LLM-Powered Answer Generation

## Objective

The objective of v0.5 is to transform the retrieval system into a true Retrieval-Augmented Generation (RAG) assistant by integrating a Large Language Model (LLM).

### New Features
- Uses the hybrid retrieval pipeline (FAISS + BM25) developed in v0.4.
- Retrieves the most relevant document chunks for a user query.
- Generates natural language answers using the TinyLlama instruction-tuned language model.
- Grounds responses using only the retrieved document context.
- Displays the supporting document sources for transparency and verification.

### Workflow

User Question
→ Hybrid Retrieval (FAISS + BM25)
→ Top Relevant Chunks
→ TinyLlama LLM
→ Natural Language Answer
→ Source References

### Learning Objectives

- Understand Retrieval-Augmented Generation (RAG) pipelines.
- Integrate an open-source instruction-tuned LLM with a document retrieval system.
- Perform context-aware answer generation from retrieved knowledge.
- Build a foundation for conversational AI systems that will be extended in future versions.

### Version Highlights

- Hybrid document retrieval
- LLM-based answer generation
- Source-grounded responses
- Interactive question-answering interface

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.9 MB/s eta 0:00:00


In [2]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


In [5]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [6]:
def chunk_text(text, chunk_size=1000, overlap=200):

    paragraphs = text.split("\n\n")

    chunks = []

    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= chunk_size:

            current_chunk += paragraph + "\n\n"

        else:

            chunks.append(current_chunk.strip())

            current_chunk = current_chunk[-overlap:] + paragraph + "\n\n"

    if current_chunk:

        chunks.append(current_chunk.strip())

    return chunks

In [7]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        chunks = chunk_text(page["text"])

        for chunk in chunks:

            all_chunks.append(
                {
                    "chunk_id": chunk_id,
                    "document": document["filename"],
                    "page": page["page"],
                    "text": chunk
                }
            )

            chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 47 chunks.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(47, 384)


In [10]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

47


In [11]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


In [12]:
def retrieve(query, top_k=5,
             semantic_weight=0.6,
             keyword_weight=0.4):

    # ======================================
    # FAISS Retrieval
    # ======================================

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    faiss_scores, faiss_indices = index.search(
        query_embedding.reshape(1, -1),
        len(all_chunks)
    )

    faiss_scores = faiss_scores[0]
    faiss_indices = faiss_indices[0]

    # ======================================
    # BM25 Retrieval
    # ======================================

    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    # ======================================
    # Normalize Scores
    # ======================================

    faiss_norm = (
        faiss_scores - faiss_scores.min()
    ) / (
        faiss_scores.max() - faiss_scores.min() + 1e-8
    )

    bm25_norm = (
        bm25_scores - bm25_scores.min()
    ) / (
        bm25_scores.max() - bm25_scores.min() + 1e-8
    )

    # ======================================
    # Hybrid Score
    # ======================================

    results = []

    for rank, idx in enumerate(faiss_indices):

        semantic = float(faiss_norm[rank])

        keyword = float(bm25_norm[idx])

        hybrid = (
            semantic_weight * semantic +
            keyword_weight * keyword
        )

        results.append({

            "document": all_chunks[idx]["document"],

            "page": all_chunks[idx]["page"],

            "chunk_id": all_chunks[idx]["chunk_id"],

            "semantic_score": semantic,

            "bm25_score": keyword,

            "hybrid_score": hybrid,

            "text": all_chunks[idx]["text"]

        })

    results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return results[:top_k]

In [23]:
query = "What algorithm was used for classification?"

results = retrieve(query)

# Keep only the best 2 chunks
context = "\n\n".join(
    [r["text"][:600] for r in results[:2]]
)

print("=" * 80)
print(context)

3.4 Data Splitting and Validation
For each classifier, the available simulation dataset was partitioned into a training set (90%
of the data) and a validation set (10%) using stratified sampling to preserve the relative class
proportions. This ensures that each class including minority classes such as prompt collapse
events is adequately represented in both subsets.
To further avoid bias from an “unusually easy” or “unusually difficult” validation set, the
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining s

The inspiral parameters used as classifier inputs total massMtot, mass ratio q, mass-weighted
tidal deformability ˜Λ, and effective inspiral spin χeff are chosen because they can be estimated
from gravitational-wave (GW) observations of the inspiral phase alone, even in cases where the
postmerger signal is too weak to detect [2]. This makes the approach suitable for low latency
classification in

In [24]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("TinyLlama Loaded")

Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama Loaded


In [29]:
def answer_question(query):

    results = retrieve(query)

    # Limit context
    context = "\n\n".join(
        [r["text"][:600] for r in results[:2]]
    )

    prompt = f"""
<|system|>
You are a helpful AI assistant.

Answer ONLY using the supplied context.

If the answer is not present,
say "I could not find the answer in the provided documents."

<|user|>

Context:

{context}

Question:

{query}

<|assistant|>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(

        **inputs,

        max_new_tokens=200,

        do_sample=False,

        repetition_penalty=1.1,

        eos_token_id=tokenizer.eos_token_id

    )

    answer = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True

    )
    answer = answer.replace(prompt, "").strip()

    return answer, results

In [30]:
answer, sources = answer_question(
    "What algorithm was used for classification?"
)

print("=" * 80)
print("ANSWER")
print("=" * 80)

print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER
The algorithm used for classification in the given text is binary model that distinguishes between: (a) Prompt collapse to a black hole (PCBH); and (b) An unspecified event type.


In [27]:
print("=" * 80)
print("REFERENCES")
print("=" * 80)

for i, source in enumerate(sources, 1):

    print()

    print(f"Source {i}")

    print(f"Document : {source['document']}")
    print(f"Page     : {source['page']}")

    print("-" * 80)

    print(source["text"][:250])

REFERENCES

Source 1
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
--------------------------------------------------------------------------------
3.4 Data Splitting and Validation
For each classifier, the available simulation dataset was partitioned into a training set (90%
of the data) and a validation set (10%) using stratified sampling to preserve the relative class
proportions. This ensure

Source 2
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 6
--------------------------------------------------------------------------------
The inspiral parameters used as classifier inputs total massMtot, mass ratio q, mass-weighted
tidal deformability ˜Λ, and effective inspiral spin χeff are chosen because they can be estimated
from gravitational-wave (GW) observations of the inspiral 

Source 3
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 9
----------------------------------------

In [28]:
while True:

    query = input("\nAsk a question (or type exit): ")

    if query.lower() == "exit":

        break

    answer, sources = answer_question(query)

    print()

    print("=" * 80)
    print("ANSWER")
    print("=" * 80)

    print(answer)

    print()

    print("=" * 80)
    print("SOURCES")
    print("=" * 80)

    for source in sources:

        print(
            f"{source['document']}  (Page {source['page']})"
        )


Ask a question (or type exit): What algorithm was used for classification?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER
The algorithm used for classification in the given text is binary model that distinguishes between: (a) Prompt collapse to a black hole (PCBH); and (b) An unspecified event type.

SOURCES
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 8)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 6)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 9)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 22)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 3)

Ask a question (or type exit): How was the dataset split?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER
The dataset was partitioned into a training set (90% of the data) and a validation set (10%). The validation set was used to ensure that each class including minority classes such as prompt collapse events is adequately represented in both subsets.

SOURCES
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 8)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 12)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 13)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 7)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 21)

Ask a question (or type exit): What is SHAP analysis?


...


ANSWER
The Event-wise probability outputs for a given parameter vector x describing the inspiral phase of a binary neutron star system are discrete probability distributions P(ωk | x), where ωk denotes the kth remnant class in the classifier's label set. The uncertainty of predictions obtained through machine learning algorithms is divided into two classes: data uncertainty (i.e., the entropy of conditional probabilities) and knowledge uncertainty (i.e., the entropy of conditional probabilities).

SOURCES
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 15)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 22)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 17)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 18)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 16)

Ask a question (or type exit): What are the Event Datasets used for Inference?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER
The Event Datasets used for inference are the simulation datasets for Classifier A and Classifier B.

SOURCES
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 6)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 8)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 15)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 3)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 22)

Ask a question (or type exit): what is GW170817 and GW190425 used for?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER
GW170817 and GW190425 are two gravitational wave events observed by the LIGO and Virgo detectors in August 2017 and April 2019, respectively. They were both detected as merging black hole binaries and are considered important discoveries in the field of gravitational wave astronomy. These events have been used to study the properties of black holes, test general relativity, and improve our understanding of the universe's early history.

SOURCES
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 4)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 15)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 3)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 16)
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf  (Page 6)

Ask a question (or type exit): exit
